In [2]:
import numpy as np

def generate_task_action(rng, T=1000, d_x=5, d_z=5, temperature=None):
    """
    Genera un singolo dataset "task-action" (coerente con l'assunzione di
    indipendenza tra azioni usata nel setting sintetico del paper):
    
    - Z ~ N(0_5, I_5): parametro latente dell'azione (5 dim, come X)
    - X_1, ..., X_T ~ N(0_5, I_5) i.i.d.: contesti
    - Y_t | X_t, Z ~ Bernoulli( sigmoid( (X_t^T Z) / temperature ) )
    
    Il fattore 'temperature' scala il prodotto scalare per evitare probabilità
    troppo estreme: dato che X_t, Z ~ N(0, I_d) indipendenti, Var(X_t^T Z) = d,
    quindi usiamo temperature = sqrt(d_x) come scala di default.
    """
    if temperature is None:
        temperature = np.sqrt(d_x)

    Z = rng.normal(size=d_z)
    X = rng.normal(size=(T, d_x))

    logits = (X @ Z) / temperature
    probs = 1.0 / (1.0 + np.exp(-logits))
    Y = rng.binomial(1, probs)

    return {"Z": Z, "X": X, "Y": Y, "probs": probs}


In [3]:
def generate_offline_dataset(n_tasks, T=1000, d_x=5, d_z=5, temperature=None, seed=0):
    """
    Genera n_tasks dataset "task-action" indipendenti, da usare come D_offline
    per il training di p_theta (Sezione 4.1 / Appendix B.2.3 del paper).
    """
    rng = np.random.default_rng(seed)
    dataset = [
        generate_task_action(rng, T=T, d_x=d_x, d_z=d_z, temperature=temperature)
        for _ in range(n_tasks)
    ]
    return dataset

In [4]:
# Iperparametri (in linea con Appendix B.2.3 del paper)
N_TASKS = 20_000
T = 1000
D_X = 5
D_Z = 5
SEED = 0

offline_data = generate_offline_dataset(
    n_tasks=N_TASKS, T=T, d_x=D_X, d_z=D_Z, seed=SEED
)

n_train = 10_000
train_data = offline_data[:n_train]
val_data = offline_data[n_train:]

print(f"Task-action totali: {len(offline_data)}")
print(f"Train: {len(train_data)} | Validation: {len(val_data)}")
print(f"Shape esempio -> Z: {train_data[0]['Z'].shape}, "
      f"X: {train_data[0]['X'].shape}, Y: {train_data[0]['Y'].shape}")
print(f"Frazione media di click (train): {np.mean([d['Y'].mean() for d in train_data]):.3f}")
print(f"esempio di elemento: {offline_data[2]}" )

Task-action totali: 20000
Train: 10000 | Validation: 10000
Shape esempio -> Z: (5,), X: (1000, 5), Y: (1000,)
Frazione media di click (train): 0.500
esempio di elemento: {'Z': array([-1.12790252,  0.59593231, -0.54588705, -0.35404405,  0.91965117]), 'X': array([[-0.25508793, -1.48439569,  0.92737964,  0.48530588, -0.63932305],
       [-0.59965484,  0.18571097, -1.97665986,  1.42201858,  1.8745738 ],
       [-0.83219933, -1.76919941, -1.62523673,  0.92892599,  0.73880947],
       ...,
       [-0.59251419, -2.51824949, -0.87023369, -0.08739605,  0.27909759],
       [-0.80503333,  0.34240096,  1.41251992,  1.35315582,  0.40327796],
       [-0.09675819, -0.36203844, -0.53785648,  0.51663471, -0.59244764]],
      shape=(1000, 5)), 'Y': array([0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 

In [5]:
import pickle
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "offline_dataset.pkl", "wb") as f:
    pickle.dump(
        {
            "train_data": train_data,
            "val_data": val_data,
            "config": {
                "n_tasks": N_TASKS,
                "T": T,
                "d_x": D_X,
                "d_z": D_Z,
                "seed": SEED,
            },
        },
        f,
    )

print(f"Dataset salvato in: {output_dir / 'offline_dataset.pkl'}")

Dataset salvato in: data\offline_dataset.pkl
